# Portugal / Millennium benchmark

Two datasets, both from the **raw Intrum CAT feed** — not PROD, which holds only 42% of
these collections. They join on `(PORTFOLIO_ID, LOAN_ID)` and cover the same 120,543 loans.
Every guard and measured figure lives in the two `.sql` headers; this notebook loads,
checks and exports them.

### `accounts` — one row per loan, 9 columns

| column | what it is |
|---|---|
| `PORTFOLIO_ID` | the batch ref, e.g. `PT050030000000` (PAM's `COMBINED_REF`) |
| `LOAN_ID` | raw Intrum `CASE_ID`. PROD's key is `4050-<LOAN_ID>` |
| `REGISTRATION_DATE` | when Intrum opened the case — **not** origination |
| `REGISTRATION_CAPITAL` | **principal** (`ORIGINAL_CAPITAL`) at registration. Complete on every loan, and the figure that reconciles to PAM. **Use this to compare** |
| `REGISTRATION_INTEREST` | interest at registration (`ORIGINAL_INTEREST`). EUR 5,974,550.16, but non-zero on only **1,044 of 120,711 loans** — incomplete, see below |
| `REGISTRATION_BALANCE` | capital **+** interest = EUR 912,333,699.57. Excludes `ACCRUED_INTEREST`, which accrues *after* registration and is 23× larger |
| `DEFAULT_DATE` | PROD's back-filled default date. 100% filled; 97.4% fall before the batch's capex month |
| `DEFAULT_DATE_SUSPECT_FLAG` | 1,876 pre-1990 dates (earliest 1978-01-01), flagged rather than nulled |
| `PRICING_SEGMENT` | `SME` if any corporate debtor on the loan, else `Individual`. Company-wins is ratified policy, not a reading of the data |

### `payments` — one row per (loan, payment date), 5 columns. All EUR.

474,900 rows over 52,062 loans that ever paid. EUR 160,045,164.05, of which
EUR 5,407,535.03 (3.38%) has no date and cannot enter a time series.

### What changed, 2026-09-10

- **Borrower type moved from PROD to raw.** PROD's `DIM_BORROWER` keys a borrower without
  the case in the key, and the resulting loss is *anti-company*, so it understated SME by
  construction. SME went **5,569 loans / EUR 279,942,246.25 → 12,844 / EUR 450,739,900.85**,
  and 167 loans PROD could not classify re-entered the population carrying **EUR 327,270.03
  of collections** previously left out. Mechanism, funnel and full reconciliation:
  `borrower_type_mapping.sql`. The `borrower_bk` grain is a data-team issue affecting every
  Intrum portfolio.
- **Output narrowed, 2026-09-11.** `HAS_COMPANY_DEBTOR` / `HAS_PRIVATE_DEBTOR` removed from
  `accounts` (the flags remain as `is_company` / `is_private` inside the query, so the
  tie-break sensitivity is a one-line restore). `CURRENCY_ID` removed from `payments` —
  474,899 of 474,900 rows are EUR and the one exception has no currency at all, so nothing
  merged and no total moved. Boolean columns renamed to a consistent `*_FLAG` suffix:
  `PAYMENT_DATE_IS_MISSING` → `MISSING_PAYMENT_DATE_FLAG` and `DEFAULT_DATE_IS_SUSPECT` →
  `DEFAULT_DATE_SUSPECT_FLAG`.
- **`REGISTRATION_BALANCE` = capital + interest**, and its interest component is incomplete
  (see the pre-export cell).
- **Insolvency dropped entirely** — a 23-month extract covering 4.87% of loans, with
  `BANKRUPTCY_END_DATE` 100% null. It cannot mark distress at acquisition; `DEFAULT_DATE` can.
- **Exclusions**: three batches with no CAT payments, and 168 loans with no borrower type in
  the raw feed (was 335 under PROD-first). Applied in **both** files so they keep pairing.

In [ ]:
import sys
sys.path.insert(0, "../../packages")   # notebook sits in work/project-elsa-benchmark/
import datalake
import pandas as pd

# Money and ratios to 5 dp, per the workspace convention. This changes only what
# is PRINTED — it never alters the values, so nothing is rounded away.
pd.set_option("display.float_format", lambda v: f"{v:,.5f}")

# Name your export here. Leave it empty and the files are written as
# accounts.csv / payments.csv; set it and they become <name>_accounts.csv etc.
EXPORT_NAME = "portugal_millennium_benchmark"

In [ ]:
# ~30 seconds. Reads the CAT case feed and classifies borrowers from raw DEBTORS.
#
# convert=False keeps Snowflake's Decimal values instead of float64. It is not
# optional here: the asserts below check CAPITAL + INTEREST == BALANCE exactly,
# and in float64 that identity fails on rounding. It is also the workspace rule
# for anything you SUM OR RECONCILE — and this is the valuation base.
# Trade-off: display.float_format does not apply to Decimal columns, so format
# money explicitly when you print it.
accounts_df = datalake.q("work/project-elsa-benchmark/accounts", convert=False)

# ⚠ convert=False also leaves DATE columns as Python date objects, and pandas
# cannot min/max/resample an object column. Coerce the dates back — this is
# safe, they carry no precision to lose. Only the money stays Decimal.
for col in ("REGISTRATION_DATE", "DEFAULT_DATE"):
    accounts_df[col] = pd.to_datetime(accounts_df[col])

print(f"{len(accounts_df):,} loans x {len(accounts_df.columns)} columns")
accounts_df.head()

In [ ]:
# ⚠ ~9 minutes. Reads every extract of a 24.7bn-row table with no date bound.
# It applies the same raw borrower classification as accounts.sql so the two
# populations match exactly — the assert below checks that they still do.
payments_df = datalake.q("work/project-elsa-benchmark/payments", convert=False)

# Same date coercion as above. Undated payments become NaT, which is what you
# want: they drop out of any time series while their money stays in the total.
payments_df["PAYMENT_DATE"] = pd.to_datetime(payments_df["PAYMENT_DATE"])

print(f"{len(payments_df):,} rows x {len(payments_df.columns)} columns")
payments_df.head()

In [ ]:
# Last payment per batch — the CAT extract cut, not the end of collections.
# Every batch stops at 2026-01-14 or earlier; PAM runs to 2026-07.
payments_df.groupby("PORTFOLIO_ID")["PAYMENT_DATE"].max()

In [ ]:
# --- sanity checks worth running before you trust either frame ---------------
# Each of these has failed at least once during development, which is why they
# are asserts and not comments.
assert accounts_df["LOAN_ID"].is_unique, "accounts must be one row per loan"
# REGISTRATION_BALANCE must be exactly its two parts — it is derived, so a
# mismatch means the query changed under you. Exact only with convert=False.
assert (accounts_df["REGISTRATION_CAPITAL"] + accounts_df["REGISTRATION_INTEREST"]
        == accounts_df["REGISTRATION_BALANCE"]).all(), \
    "REGISTRATION_BALANCE must equal CAPITAL + INTEREST row by row"
assert set(accounts_df["PRICING_SEGMENT"]) == {"Individual", "SME"}, \
    "pricing_segment should only ever be Individual or SME — Unknown is excluded in the .sql"
assert set(payments_df["LOAN_ID"]) <= set(accounts_df["LOAN_ID"]), \
    "payments must not contain a loan that accounts drops — the two exclusions have drifted apart"

total = payments_df["TOTAL_COLLECTIONS"].sum()
undated = payments_df.loc[payments_df["MISSING_PAYMENT_DATE_FLAG"], "TOTAL_COLLECTIONS"].sum()

print(f"portfolios           {accounts_df['PORTFOLIO_ID'].nunique():>12,}   (expect 11)")
print(f"loans                {len(accounts_df):>12,}   (expect 120,543)")
print(f"payment rows         {len(payments_df):>12,}   (expect 474,900)")
print(f"collections total    {total:>12,.2f}   (expect 160,045,164.05)")
print(f"  of which undated   {undated:>12,.2f}   ({undated/total:.2%} — no date, cannot be placed in a month)")
print(f"loans that ever paid {payments_df['LOAN_ID'].nunique():>12,}   (expect 52,062)")
print()
# Segment split on both bases. Formatted explicitly because display.float_format
# does not reach Decimal columns — a raw print shows 18 decimal places.
_cap = accounts_df["REGISTRATION_CAPITAL"].sum()
_bal = accounts_df["REGISTRATION_BALANCE"].sum()
print(f"{'':11}{'loans':>9}{'capital':>18}{'balance':>18}{'% of bal':>10}")
for _seg, _g in accounts_df.groupby("PRICING_SEGMENT"):
    _c, _b = _g["REGISTRATION_CAPITAL"].sum(), _g["REGISTRATION_BALANCE"].sum()
    print(f"{_seg:11}{len(_g):>9,}{_c:>18,.2f}{_b:>18,.2f}{_b / _bal * 100:>9.2f}%")
print(f"{'TOTAL':11}{len(accounts_df):>9,}{_cap:>18,.2f}{_bal:>18,.2f}{100:>9.2f}%")
print(f"\ninterest {accounts_df['REGISTRATION_INTEREST'].sum():,.2f} "
      f"is IN balance and NOT in capital — 1,044 loans carry any (see below)")

# ⚠ DO NOT compare `collections total` straight to PAM's lifetime figure of
# 160,422,487.54. This file is a narrower population by design — see the
# markdown cell below — and the four period differences in pam_cat_match.sql
# apply on top. Use work/.archive/2026-10-02-benchmark-investigation-trail/pam_cat_match.sql for a real
# CAT-vs-PAM reconciliation; this block only checks the frames are intact.
#
# Filter on PAYMENT_DATE for any time series — but expect the total to fall
# short by the undated figure above. Do not filter them out and then quote the
# remainder as the book's collections.

## ⚠ Read before exporting — everything worth noting, in one place

### The numbers this run should produce

| | count | EUR |
|---|---|---|
| `accounts` loans | 120,543 | |
| &nbsp;&nbsp;`REGISTRATION_CAPITAL` — principal, complete | | 906,359,149.41 |
| &nbsp;&nbsp;`REGISTRATION_INTEREST` — **incomplete**, 1,044 loans | | 5,974,550.16 |
| &nbsp;&nbsp;`REGISTRATION_BALANCE` — the two added | | 912,333,699.57 |
| &nbsp;&nbsp;SME (49.41%) | 12,844 | 450,739,900.85 |
| &nbsp;&nbsp;Individual (50.59%) | 107,699 | 461,593,798.72 |
| `payments` rows / paying loans | 474,900 / 52,062 | 160,045,164.05 |
| &nbsp;&nbsp;of which **undated** (3.38%) | | 5,407,535.03 |

Lifetime recovery: **17.658%** on capital · 17.542% on balance · 18.106% on PAM principal.
On dated cash only it is 17.061%.

### Say these five things whenever you quote a figure

1. **Which balance column.** Capital and balance differ by EUR 5,974,550.16.
2. **The balance is a floor.** Its interest is non-zero on 0.86% of loans, while 97.7% of the
   zero-interest loans had defaulted a median **3.01 years** before registration. Three years
   in default cannot owe zero interest, so the zeros are non-delivery. Use
   `REGISTRATION_CAPITAL` for anything like-for-like.
3. **`SME` means "a corporate debtor is on the loan"** — not a size class. There is no income,
   turnover, headcount or company number in the feed. And 11,374 loans (EUR 398,562,465.78,
   88% of the SME balance) are SME **by ratified policy**, not evidence; under the opposite
   rule SME is 1,470 loans / EUR 52,177,435.07.
4. **Which recovery denominator** — the three above span 0.56pp portfolio-wide, but
   `PT050030000000` alone reads **40.44% or 50.97%** depending on the choice.
5. **This is not "collections to date."** CAT stops at its extract date 2026-01-14; PAM runs
   to 2026-07. Six months and EUR 1,653,076.48 are missing — 1.2% of a 21-year lifetime but
   **54% of a calendar year**. Read anything recent from PAM.

### Never

- Filter out the undated cash and then quote the remainder as the book's collections.
- Join `accounts` to `payments` without aggregating payments to loan level **first** — and
  use a LEFT join, or you drop the 68,481 loans that never paid.
- Reuse `payments` outside PT / Millennium without re-reading note N5. Currency was dropped
  from the grain because this scope is 100% EUR; elsewhere it would sum across currencies
  silently.
- Use PROD's `DIM_BORROWER` for the segment, or "recover" its missing loans by dropping
  `IS_CURRENT` — same borrowed values, plus fan-out.
- Quote a CAT-vs-PAM ratio without stating the period cut. The headline 99.88% is a
  coincidence of two errors cancelling.
- Quote an insolvency rate (23-month extract, 4.87% of loans) or a payment count (43,658
  loans carry amount-less rows). Money is unaffected by both.
- Put `SME` into anything regulatory or capital-related.

### Open with Intrum — written for someone with no context

Intrum sends us two separate feeds: **CAT**, the loan-by-loan servicing system, and **PAM**,
a batch-level finance summary. This dataset is built from CAT. The items below are places
where the two disagree, or where CAT looks incomplete. None of them blocks a
portfolio-level valuation; all of them are Intrum's to explain, not ours to reconcile away.

| # | The issue, in plain terms | EUR at stake |
|---|---|---|
| 1 | **Interest that was never delivered.** Nearly every loan here defaulted years before Intrum took it on, yet 88% of them show *zero* unpaid interest — both at takeover and accrued since. Interest that real can't be zero. We are almost certainly understating what the borrowers owe. | unknown, but **≫ 5,974,550.16** |
| 2 | **`PT15018A040000` — we paid for loans that aren't there.** Finance says this batch cost EUR 18.6m. The servicing system holds 84 loans worth EUR 3.5m. The rest has no loans behind it at all. | **15,099,054.38** |
| 3 | **`PT15018G000000` — 2,173 loans lost their batch label.** 12% of this batch has had its batch reference wiped in Intrum's system. We reassign them by last-known batch, which is a defensible guess, not a fact. Their collections ride on that guess. | **6,845,160.28** |
| 4 | **`PT050030000000` — the two feeds disagree by EUR 6.2m.** We checked our side: the loans are real and not double-counted, so it's the finance figure that needs explaining. It matters because this batch's recovery rate reads **40% or 51%** depending which number you divide by. | **6,207,347.82** |
| 5 | **`PT08018B000000` — same kind of disagreement, never looked at.** Could be benign, could be issue 4 again. Nobody has checked. | **2,531,611.08** |
| 6 | **We're six months behind.** CAT's data stops 2026-01-14; finance runs to 2026-07. Only 1.2% of a 21-year history — but **54% of this calendar year**, so never use this for a current view. | **1,653,076.48** |
| 7 | **EUR 1m of "collections" that probably never happened.** Two batches each report a single month of collections exactly equal to the full price we paid for them, in the month we bought them. Real collections never look like that — it reads as an accounting entry booked into the cash column. | **1,000,000.00** |
| 8 | **Nobody has confirmed what the borrower codes mean.** The entire SME / Individual split rests on reading `C` as company and `P` as private. It's the obvious reading and the data team reads it the same way — but every borrower name field is empty, so nothing in the feed proves it. This is the single assumption everything else sits on. | — |

**Separately, for our own data team (not Intrum):** the borrower key in
`int_npl_borrower_intrum.sql` omits the case, which is what forced this dataset off the
warehouse and onto the raw feed. It affects every Intrum portfolio, not just Portugal.

Full detail: [work/intrum-escalations.md](../../work/intrum-escalations.md). Each `.sql`
header carries its own guards and every figure above.

In [ ]:
accounts_df = pd.read_csv('/Users/sharlenechin/data-warehouse/local-workspace/work/project-elsa-benchmark/portugal_millennium_benchmark_accounts.csv')
payments_df = pd.read_csv('/Users/sharlenechin/data-warehouse/local-workspace/work/project-elsa-benchmark/portugal_millennium_benchmark_payments.csv')

In [ ]:
# --- portfolio summary, one row per batch -------------------------------------
# The acquisition-side table: what Intrum says it bought, beside what we hold,
# with a line on every batch saying why it is in or out.
#
# ⚠ THE ACQUISITION COLUMNS COME FROM PAM, NOT FROM accounts_df. Capex, price
# and acquired balance are deal-level facts only PAM carries; a roll-up of our
# loan-level data is a different quantity and is shown separately below.
pam_df = datalake.q("work/project-elsa-benchmark/portfolio_summary", convert=False)

# ⚠ BOTH DATE COLUMNS MUST BE DATETIMES BEFORE THE RULES BELOW RUN. Cells 2-3
# coerce them after datalake.q, but reloading the exported CSV with a plain
# pd.read_csv (no parse_dates=) brings them back as STRINGS, and then:
#   · .dt.year raises AttributeError, and
#   · DEFAULT_DATE > REGISTRATION_DATE silently becomes a STRING comparison,
#     which happens to agree for ISO dates and would not for any other format.
# Coercing here costs nothing and makes the cell correct from either source.
for _col in ("REGISTRATION_DATE", "DEFAULT_DATE"):
    accounts_df[_col] = pd.to_datetime(accounts_df[_col])

# The three batches accounts.sql and payments.sql drop, and why. Kept as text
# here so the summary can state the reason rather than leave a blank row.
EXCLUDED_BATCH_REASONS = {
    "PT06014B010000": "EXCLUDED, whole batch — no cases in CAT under any source system; a feed "
                      "gap, not an empty book. Its single PAM month equals its own acquisition "
                      "principal to the cent: an accounting entry, not cash.",
    "PT070040000000": "EXCLUDED, whole batch — 2 loans holding EUR 2.00 of capital each, "
                      "registered 279 days after capex where every other batch lands within -2 "
                      "to +40. Never paid. Its single PAM month is the same accounting entry.",
    "PT15018A040000": "EXCLUDED, whole batch — its 84 cases resolve to PT16023C000000 under "
                      "latest-in-scope-wins, so it was never in the output. PAM's acquired "
                      "principal has no loans behind it; second-largest open Intrum item.",
}

# What stages/01-cleaning drops downstream of THIS export, recomputed from the
# same three rules rather than quoted, so the counts cannot drift from the code.
#   zero_balance    registration_balance <= 0        -> all 7 are exactly 0.00;
#                                                       no balance in this book is negative
#   not_in_default  default_date > registration_date -> 413, of which 405 on PT15018G000000
#   excluded_year   registration year in [2014]      -> 1, a healthy loan dropped only because
#                                                       it is a one-account vintage
# ⚠ THE THREE RULES DO NOT OVERLAP: 7 + 413 + 1 = 421 DISTINCT accounts, so the
# cleaned population is 120,122 — NOT the 120,123 quoted in the data room and
# drawn in the vintage chart, both of which predate the 2014 rule. Whoever
# consumes those needs telling.
EXCLUSION_RULES = {
    "zero balance":                  lambda df_: df_["REGISTRATION_BALANCE"].le(0),
    "not in default at registration": lambda df_: df_["DEFAULT_DATE"].gt(df_["REGISTRATION_DATE"]),
    "registering in 2014":           lambda df_: df_["REGISTRATION_DATE"].dt.year.eq(2014),
}

# One row per (batch, rule) — the itemised answer to "what was removed and why".
portfolio_exclusions_df = (
    pd.concat(
        [accounts_df.loc[test(accounts_df)].assign(_rule=rule)
         for rule, test in EXCLUSION_RULES.items()]
    )
    .groupby(["PORTFOLIO_ID", "_rule"])
    .agg(accounts=("LOAN_ID", "size"), balance=("REGISTRATION_BALANCE", "sum"))
    .reset_index()
    .rename(columns={"PORTFOLIO_ID": "Intrum Portfolio ID", "_rule": "Exclusion Rule",
                     "accounts": "Accounts", "balance": "Balance"})
    .sort_values(["Intrum Portfolio ID", "Exclusion Rule"], ignore_index=True)
    # Balance arrives as Decimal (convert=False), and display.float_format does
    # not reach Decimal — it prints as 0E-18. Cast for display; the sum above
    # was already exact, so nothing is lost here.
    .astype({"Accounts": "int64", "Balance": "float64"})
)

_dropped = pd.concat([test(accounts_df) for test in EXCLUSION_RULES.values()], axis=1).any(axis=1)

ours = accounts_df.assign(
    _dropped=_dropped,
    _debt_age_years=(accounts_df["REGISTRATION_DATE"]
                     - accounts_df["DEFAULT_DATE"]).dt.days / 365.25,
).groupby("PORTFOLIO_ID").agg(
    our_loans=("LOAN_ID", "size"),
    our_balance=("REGISTRATION_BALANCE", "sum"),       # Decimal in, exact sum
    our_debt_age=("_debt_age_years", "mean"),
    accounts_excluded=("_dropped", "sum"),
)
ours["our_collections"] = payments_df.groupby("PORTFOLIO_ID")["TOTAL_COLLECTIONS"].sum()
ours["our_loans_after_cleaning"] = ours["our_loans"] - ours["accounts_excluded"]

# LEFT from PAM, so all 14 batches appear and the 3 we drop show as unmatched
# rather than silently vanishing — that absence is the point of the table.
portfolio_detail_df = pam_df.set_index("PORTFOLIO_ID").join(ours).reset_index()

# Decimal only mattered through the summation; float from here is for display
# and for the one ratio. Money is never rounded, only formatted.
for _col in ("CAPEX_AMOUNT", "ACQ_BALANCE", "PRICE_PCT", "AVG_DEBT_AGE_YEARS",
             "PAM_COLLECTIONS", "our_balance", "our_collections"):
    portfolio_detail_df[_col] = portfolio_detail_df[_col].astype(float)

# ⚠ COUNTS MUST STAY COUNTS. The LEFT join puts NaN on the three excluded
# batches, and NaN is a float, so every integer column silently promotes —
# "406 accounts excluded" renders as 406.00000. Int64 is pandas' NULLABLE
# integer: <NA> where a batch has no loans, a whole number everywhere else.
for _col in ("CAPEX_YEAR", "ACQ_ACCOUNTS", "ACQ_DEBTORS", "our_loans",
             "accounts_excluded", "our_loans_after_cleaning"):
    portfolio_detail_df[_col] = portfolio_detail_df[_col].astype("Int64")

_drops_by_batch = (
    portfolio_exclusions_df
    .assign(text=lambda df_: df_["Accounts"].astype(int).astype(str) + " " + df_["Exclusion Rule"])
    .groupby("Intrum Portfolio ID")["text"].apply("; ".join)
)


def _commentary(row):
    """One line per batch: in or out, and exactly what cleaning removes from it."""
    batch = row["Intrum Portfolio ID"]
    if batch in EXCLUDED_BATCH_REASONS:
        return EXCLUDED_BATCH_REASONS[batch]
    drops = _drops_by_batch.get(batch)
    parts = [f"Included — {int(row['our_loans']):,} loans exported, "
             f"EUR {row['our_balance']:,.2f} registration balance."]
    parts.append(f"Cleaning then drops {drops}, leaving "
                 f"{int(row['our_loans_after_cleaning']):,}." if drops
                 else "Cleaning drops nothing; all loans survive.")
    # PAM's own AVERAGE_DEBT_AGE is 0.00 on PT15018G000000 — not a real zero.
    if row["AVG_DEBT_AGE_YEARS"] == 0:
        parts.append("⚠ PAM reports 0.00 average debt age; ours is shown instead.")
    return " ".join(parts)


portfolio_detail_df = portfolio_detail_df.rename(columns={
    "PORTFOLIO_ID": "Intrum Portfolio ID",
    "CAPEX_YEAR":   "Capex Year",
    "CAPEX_AMOUNT": "Capex Amount",
    "PRICE_PCT":    "Price %",
    "ACQ_BALANCE":  "Acq Balance",
    "ACQ_ACCOUNTS": "Acq Accounts",
    "PAM_COLLECTIONS": "Collections",
})
# PAM's figure, except where it reports an impossible zero.
portfolio_detail_df["Avg. Debt Age"] = portfolio_detail_df["AVG_DEBT_AGE_YEARS"].where(
    portfolio_detail_df["AVG_DEBT_AGE_YEARS"] > 0, portfolio_detail_df["our_debt_age"])
portfolio_detail_df["Accounts Excluded"] = portfolio_detail_df["accounts_excluded"]
portfolio_detail_df["Commentary / Exclusions"] = portfolio_detail_df.apply(_commentary, axis=1)

portfolio_summary_df = portfolio_detail_df[[
    "Intrum Portfolio ID", "Capex Year", "Capex Amount", "Price %", "Acq Balance",
    "Acq Accounts", "Avg. Debt Age", "Collections", "Accounts Excluded",
    "Commentary / Exclusions",
]]

# ⚠ Acq Accounts is PAM's ACQCASES, counted at acquisition. It is NOT our loan
# count and the two must never be differenced — PT050030000000 reads 42,011
# here against 28,312 loans in accounts_df. Both are in portfolio_detail_df.
# ⚠ Collections is PAM's, running to 2026-08. Ours stops at the CAT extract,
# 2026-01-14, and is in portfolio_detail_df as `our_collections`.
_excluded = int(portfolio_exclusions_df["Accounts"].sum())
print(f"{len(portfolio_summary_df)} PAM batches, "
      f"{int(portfolio_detail_df['our_loans'].notna().sum())} of them in our datasets")
print(f"\nexcluded accounts, by rule — {_excluded} distinct, no loan caught by two rules:")
print(portfolio_exclusions_df.to_string(index=False))
print(f"\n{len(accounts_df):,} exported - {_excluded} = "
      f"{len(accounts_df) - _excluded:,} after cleaning")
print("⚠ the data room and the vintage chart both say 120,123 — they predate the "
      "2014 rule and are one account behind.")
portfolio_summary_df

In [ ]:
# --- export ------------------------------------------------------------------
stem = f"{EXPORT_NAME}_" if EXPORT_NAME else ""
for name, df in (("accounts", accounts_df), ("payments", payments_df)):
    path = f"{stem}{name}.csv"
    df.to_csv(path, index=False)
    print(f"{len(df):>9,} rows -> {path}")